In [33]:
# Prepare the final company dataset
#
# Read the populated Bloomberg workbooks recursively from data/processed.
# Preserve every company in the security list and produce one row per company.
# All explanatory text uses Python comments, following notebooks 01 and 02.
#
# Annual and market histories are calculated before selecting three snapshots.
# Extra history supplies trend baselines; only the requested snapshots appear
# in the final wide dataset. Missing values and Bloomberg errors remain NaN.
# No imputation, winsorization, scaling, or company removal is performed.
#
# IMPORTANT: Bloomberg observation dates are not verified release dates.
# This dataset is aligned by recorded dates, not a point-in-time vintage.
# Optional publication delays below can approximate availability but cannot
# remove revisions or establish historical release dates.
#
# credit_spread retains the requested alias, but LUACTRUU / PX_LAST is a
# corporate-bond total-return index, not a yield spread.
# Source: https://assets.bbhub.io/professional/sites/27/US-Corporate-Index.pdf

In [34]:
from pathlib import Path
from numbers import Real
import ast
import json
import warnings

import numpy as np
import pandas as pd
from openpyxl import load_workbook

ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks").is_dir() and (path / "data").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or a folder inside it.")

INPUT_DIR = ROOT / "data/processed"
SECURITY_LIST = ROOT / "data/intermediate/bloomberg_security_list.csv"
OUTPUT_DIR = ROOT / "data/clean"

# lag_1y = latest snapshot by prediction_date; lag_2y and lag_3y use cutoffs
# one and two years earlier. Set LAG_OFFSETS_YEARS = {1: 1, 2: 2, 3: 3}
# instead for literal one-, two-, and three-year-before-prediction snapshots.
LAG_OFFSETS_YEARS = {1: 0, 2: 1, 3: 2}

# Defaults use Bloomberg's recorded observation dates. Delays are explicit
# assumptions, not verified publication dates. They can be increased here.
ANNUAL_PUBLICATION_DELAY_DAYS = 0
MACRO_PUBLICATION_DELAY_DAYS = {
    "inflation_cpi_yoy": 0, "real_gdp_growth": 0, "unemployment_rate": 0,
    "effective_federal_funds_rate": 0, "treasury_3m_rate": 0,
    "treasury_10y_rate": 0, "credit_spread": 0, "vix": 0,
}
# Avoid substituting arbitrarily old statements for missing fiscal years.
MAX_ANNUAL_AGE_DAYS = 550
MIN_ANNUAL_GAP_DAYS = 300
MAX_ANNUAL_GAP_DAYS = 430
# Complete calendar months are used for comparable market/macroeconomic windows.
# No missing market months are forward-filled. GDP can carry through its quarter.
MACRO_MAX_AGE_DAYS = {"D": 10, "M": 62, "Q": 125}

In [35]:
ANNUAL_TREND_METHOD = {
    "total_assets": "pct_change",
    "total_liabilities": "pct_change",
    "total_debt": "pct_change",
    "short_term_debt": "pct_change",
    "long_term_debt": "pct_change",
    "cash_short_term_investments": "pct_change",
    "current_assets": "pct_change",
    "current_liabilities": "pct_change",
    "shareholders_equity": "pct_change",
    "accounts_receivable": "pct_change",
    "inventory": "pct_change",
    "working_capital": "absolute_change",
    "revenue": "pct_change",
    "ebit": "absolute_change",
    "ebitda": "absolute_change",
    "net_income": "absolute_change",
    "interest_expense": "pct_change",
    "depreciation_amortization": "pct_change",
    "operating_cash_flow": "absolute_change",
    "capital_expenditures": "pct_change",
    "free_cash_flow": "absolute_change",
}

DERIVED_FINANCIAL_FORMULAS = {
    # Leverage
    "debt_to_assets": "total_debt / total_assets",
    "liabilities_to_assets": "total_liabilities / total_assets",
    "debt_to_equity": "total_debt / shareholders_equity",
    "liabilities_to_equity": "total_liabilities / shareholders_equity",
    "short_term_debt_to_total_debt": "short_term_debt / total_debt",
    "long_term_debt_to_total_debt": "long_term_debt / total_debt",
    # Liquidity
    "current_ratio": "current_assets / current_liabilities",
    "quick_ratio": "(current_assets - inventory) / current_liabilities",
    "cash_ratio": "cash_short_term_investments / current_liabilities",
    "cash_to_assets": "cash_short_term_investments / total_assets",
    "working_capital_to_assets": "working_capital / total_assets",
    "working_capital_to_liabilities": "working_capital / total_liabilities",
    # Profitability
    "ebit_margin": "ebit / revenue",
    "ebitda_margin": "ebitda / revenue",
    "net_income_margin": "net_income / revenue",
    "return_on_assets": "net_income / total_assets",
    "return_on_equity": "net_income / shareholders_equity",
    "ebit_to_assets": "ebit / total_assets",
    "ebitda_to_assets": "ebitda / total_assets",
    # Cash flow strength
    "operating_cash_flow_to_assets": "operating_cash_flow / total_assets",
    "operating_cash_flow_to_liabilities": "operating_cash_flow / total_liabilities",
    "operating_cash_flow_to_debt": "operating_cash_flow / total_debt",
    "free_cash_flow_to_assets": "free_cash_flow / total_assets",
    "free_cash_flow_to_debt": "free_cash_flow / total_debt",
    # Debt service
    "interest_coverage": "ebit / interest_expense",
    "ebitda_interest_coverage": "ebitda / interest_expense",
    "cash_flow_interest_coverage": "operating_cash_flow / interest_expense",
    "debt_to_ebitda": "total_debt / ebitda",
    "debt_to_operating_cash_flow": "total_debt / operating_cash_flow",
    # Operating efficiency
    "asset_turnover": "revenue / total_assets",
    "receivables_to_revenue": "accounts_receivable / revenue",
    "inventory_to_revenue": "inventory / revenue",
    "capital_expenditure_to_assets": "capital_expenditures / total_assets",
    "capital_expenditure_to_revenue": "capital_expenditures / revenue",
    # Market-based ratios: market cap at the same snapshot's complete month.
    "market_to_book": "market_cap / shareholders_equity",
    "book_to_market": "shareholders_equity / market_cap",
    "market_cap_to_assets": "market_cap / total_assets",
    "enterprise_value_to_assets": "(market_cap + total_debt - cash_short_term_investments) / total_assets",
    "enterprise_value_to_ebitda": "(market_cap + total_debt - cash_short_term_investments) / ebitda",
}

MARKET_TREND_FEATURES = {
    field: {"1y": 12, "2y": 24, "3y": 36}
    for field in ["adjusted_price", "market_cap", "shares_outstanding", "volume"]
}
# Retain the additional original fields collected in notebook 02.
MARKET_FIELDS = [
    "adjusted_price", "market_cap", "shares_outstanding", "beta",
    "volume", "turnover", "bid_ask_spread",
]
MARKET_DERIVED_FEATURES = {
    "monthly_return": "adjusted_price.pct_change(1)",
    "return_3m": "adjusted_price.pct_change(3)",
    "return_6m": "adjusted_price.pct_change(6)",
    "return_12m": "adjusted_price.pct_change(12)",
    "volatility_3m": "monthly_return.rolling(3).std() * sqrt(12)",
    "volatility_6m": "monthly_return.rolling(6).std() * sqrt(12)",
    "volatility_12m": "monthly_return.rolling(12).std() * sqrt(12)",
    "rolling_max_price_12m": "adjusted_price.rolling(12).max()",
    "drawdown_12m": "adjusted_price / rolling_max_price_12m - 1",
    "maximum_drawdown_12m": "drawdown_12m.rolling(12).min()",
    "average_volume_12m": "volume.rolling(12).mean()",
    "volume_volatility_12m": "volume.rolling(12).std()",
    "average_market_cap_12m": "market_cap.rolling(12).mean()",
    "market_cap_growth_12m": "market_cap.pct_change(12)",
}

MACRO_FIELDS = {
    "inflation_cpi_yoy": {"security": "CPI YOY Index", "field": "PX_LAST", "periodicity": "M"},
    "real_gdp_growth": {"security": "GDP CQOQ Index", "field": "PX_LAST", "periodicity": "Q"},
    "unemployment_rate": {"security": "USURTOT Index", "field": "PX_LAST", "periodicity": "M"},
    "effective_federal_funds_rate": {"security": "FEDL01 Index", "field": "PX_LAST", "periodicity": "D"},
    "treasury_3m_rate": {"security": "USGG3M Index", "field": "PX_LAST", "periodicity": "D"},
    "treasury_10y_rate": {"security": "USGG10YR Index", "field": "PX_LAST", "periodicity": "D"},
    "credit_spread": {"security": "LUACTRUU Index", "field": "PX_LAST", "periodicity": "M"},
    "vix": {"security": "VIX Index", "field": "PX_LAST", "periodicity": "D"},
}
MACRO_DERIVED_FEATURES = {
    "yield_curve_slope": "treasury_10y_rate - treasury_3m_rate",
    "yield_curve_inverted": "treasury_10y_rate < treasury_3m_rate",
    "real_policy_rate": "effective_federal_funds_rate - inflation_cpi_yoy",
    "credit_spread_change": "credit_spread.diff()",
    "vix_change": "vix.diff()",
}

In [36]:
# Read paired Bloomberg Date/Value columns
#
# Bloomberg errors, unresolved formulas, and nonnumeric cells become NaN.
# Dates are parsed independently for every field; horizontal blocks need not
# have equal row counts. Duplicate dates with conflicting values stop the run.
# No zero or missing-value substitution is used to create financial data.

In [37]:
def source_date(value):
    if value is None or pd.isna(value):
        return pd.NaT
    if isinstance(value, Real) and not isinstance(value, bool):
        # Excel serial dates, including dates stored without a number format.
        if 1 <= value <= 100000:
            return pd.Timestamp("1899-12-30") + pd.to_timedelta(float(value), unit="D")
        return pd.NaT
    if isinstance(value, str) and (value.startswith("#") or value.startswith("=")):
        return pd.NaT
    try:
        result = pd.to_datetime(value, errors="coerce")
        return pd.Timestamp(result).normalize() if pd.notna(result) else pd.NaT
    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def parse_field(rows, field, context, quality):
    headers = list(rows[0]) if rows else []
    date_name, value_name = f"{field}_Date", f"{field}_Value"
    record = {**context, "field": field}
    if date_name not in headers or value_name not in headers:
        quality.append({**record, "status": "missing_columns", "valid_values": 0})
        return pd.Series(dtype=float, index=pd.DatetimeIndex([], name="date"), name=field)
    di, vi = headers.index(date_name), headers.index(value_name)
    raw = pd.DataFrame(
        [(row[di], row[vi]) for row in rows[1:] if row[di] is not None or row[vi] is not None],
        columns=["raw_date", "raw_value"],
    )
    if raw.empty:
        quality.append({**record, "status": "empty", "valid_values": 0})
        return pd.Series(dtype=float, index=pd.DatetimeIndex([], name="date"), name=field)
    raw["date"] = raw["raw_date"].map(source_date)
    raw["value"] = pd.to_numeric(raw["raw_value"], errors="coerce").astype(float)
    raw["value"] = raw["value"].replace([np.inf, -np.inf], np.nan)
    error_tokens = sorted({
        str(v) for v in [*raw["raw_date"], *raw["raw_value"]]
        if isinstance(v, str) and v.startswith(("#", "="))
    })
    valid_dates = raw.loc[raw["date"].notna(), ["date", "value"]].copy()
    duplicate_count = int(valid_dates.duplicated("date").sum())
    if duplicate_count:
        conflicts = valid_dates.groupby("date")["value"].nunique(dropna=False).gt(1)
        if conflicts.any():
            raise ValueError(f"Conflicting duplicate dates: {record}; {conflicts[conflicts].index.tolist()}")
        valid_dates = valid_dates.drop_duplicates("date")
    series = valid_dates.set_index("date")["value"].sort_index().rename(field)
    quality.append({
        **record,
        "status": "ok" if series.notna().any() else "no_numeric_values",
        "source_rows": len(raw),
        "invalid_date_rows": int(raw["date"].isna().sum()),
        "missing_or_invalid_values": int(raw["value"].isna().sum()),
        "duplicate_date_rows": duplicate_count,
        "valid_values": int(series.notna().sum()),
        "first_date": series.index.min(), "last_date": series.index.max(),
        "error_examples": " | ".join(error_tokens[:5]),
    })
    return series


def safe_divide(numerator, denominator):
    if isinstance(denominator, pd.Series):
        result = numerator / denominator.where(denominator.ne(0))
    elif pd.isna(denominator) or denominator == 0:
        return numerator * np.nan
    else:
        result = numerator / denominator
    if isinstance(result, pd.Series):
        return result.replace([np.inf, -np.inf], np.nan)
    return float(result) if pd.notna(result) and np.isfinite(result) else np.nan


def financial_formula(expression, values):
    # Only arithmetic and named inputs are allowed; no Python eval.
    def visit(node):
        if isinstance(node, ast.Name):
            return values[node.id]
        if isinstance(node, ast.BinOp):
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Add):
                return left + right
            if isinstance(node.op, ast.Sub):
                return left - right
            if isinstance(node.op, ast.Mult):
                return left * right
            if isinstance(node.op, ast.Div):
                return safe_divide(left, right)
        raise ValueError(f"Unsupported formula: {expression}")
    return visit(ast.parse(expression, mode="eval").body)


def completed_month(cutoff):
    cutoff = pd.Timestamp(cutoff).normalize()
    period = cutoff.to_period("M")
    return period if period.end_time.normalize() <= cutoff else period - 1


def annual_history(series_by_field):
    frame = pd.concat([series_by_field[f] for f in ANNUAL_TREND_METHOD], axis=1).sort_index()
    for field, method in ANNUAL_TREND_METHOD.items():
        series = series_by_field[field].sort_index()
        gap = series.index.to_series().diff().dt.days
        consecutive_years = gap.between(MIN_ANNUAL_GAP_DAYS, MAX_ANNUAL_GAP_DAYS)
        if method == "pct_change":
            change = safe_divide(series, series.shift(1)) - 1
        else:
            change = series.diff()
        frame[f"{field}_{method}"] = change.where(consecutive_years).reindex(frame.index)
    return frame.rename_axis("date")


def monthly_history(series_by_field, start_date, end_date):
    months = pd.period_range(pd.Timestamp(start_date).to_period("M"),
                             pd.Timestamp(end_date).to_period("M"), freq="M")
    frame = pd.DataFrame(index=months, columns=MARKET_FIELDS, dtype=float)
    for field in MARKET_FIELDS:
        series = series_by_field[field].sort_index()
        if not series.empty:
            # tail preserves a missing final observation instead of taking an
            # earlier nonmissing value from the same month.
            last = series.groupby(series.index.to_period("M")).tail(1)
            last.index = last.index.to_period("M")
            frame[field] = last.reindex(months)
    for field, windows in MARKET_TREND_FEATURES.items():
        for window, periods in windows.items():
            frame[f"{field}_pct_change_{window}"] = frame[field].pct_change(periods, fill_method=None)
    price = frame["adjusted_price"]
    frame["monthly_return"] = price.pct_change(1, fill_method=None)
    for periods in [3, 6, 12]:
        frame[f"return_{periods}m"] = price.pct_change(periods, fill_method=None)
        frame[f"volatility_{periods}m"] = (
            frame["monthly_return"].replace([np.inf, -np.inf], np.nan)
            .rolling(periods, min_periods=periods).std(ddof=1) * np.sqrt(12)
        )
    frame["rolling_max_price_12m"] = price.rolling(12, min_periods=12).max()
    frame["drawdown_12m"] = safe_divide(price, frame["rolling_max_price_12m"]) - 1
    # Implement the requested two-stage rolling formula exactly. This is not
    # a separate peak-to-trough optimization inside a single 12-month window.
    frame["maximum_drawdown_12m"] = frame["drawdown_12m"].rolling(12, min_periods=12).min()
    frame["average_volume_12m"] = frame["volume"].rolling(12, min_periods=12).mean()
    frame["volume_volatility_12m"] = frame["volume"].rolling(12, min_periods=12).std(ddof=1)
    frame["average_market_cap_12m"] = frame["market_cap"].rolling(12, min_periods=12).mean()
    frame["market_cap_growth_12m"] = frame["market_cap"].pct_change(12, fill_method=None)
    return frame.replace([np.inf, -np.inf], np.nan).rename_axis("month")


def macro_monthly_history(series_by_field, start_date, end_date):
    months = pd.period_range(pd.Timestamp(start_date).to_period("M"),
                             pd.Timestamp(end_date).to_period("M"), freq="M")
    dates = months.to_timestamp("M")
    frame = pd.DataFrame(index=months)
    provenance = pd.DataFrame(index=months)
    for field, config in MACRO_FIELDS.items():
        series = series_by_field[field]
        if series.empty:
            frame[field] = np.nan
            provenance[field] = pd.NaT
            continue
        source = series.rename("value").reset_index()
        source.columns = ["source_date", "value"]
        source["available_date"] = source["source_date"] + pd.to_timedelta(
            MACRO_PUBLICATION_DELAY_DAYS[field], unit="D"
        )
        matched = pd.merge_asof(
            pd.DataFrame({"month_end": dates}),
            source.sort_values("available_date"), left_on="month_end",
            right_on="available_date", direction="backward",
            tolerance=pd.Timedelta(days=MACRO_MAX_AGE_DAYS[config["periodicity"]]),
        )
        frame[field] = matched["value"].to_numpy()
        provenance[field] = matched["source_date"].to_numpy()
    frame["yield_curve_slope"] = frame["treasury_10y_rate"] - frame["treasury_3m_rate"]
    valid_curve = frame[["treasury_10y_rate", "treasury_3m_rate"]].notna().all(axis=1)
    frame["yield_curve_inverted"] = (frame["treasury_10y_rate"] < frame["treasury_3m_rate"]).astype(float).where(valid_curve)
    frame["real_policy_rate"] = frame["effective_federal_funds_rate"] - frame["inflation_cpi_yoy"]
    # Both changes are month-over-month differences of month-end levels.
    frame["credit_spread_change"] = frame["credit_spread"].diff()
    frame["vix_change"] = frame["vix"].diff()
    return frame.rename_axis("month"), provenance.rename_axis("month")

In [38]:
# Load company metadata and populated workbooks
#
# Security_List identifies companies; sheet prefixes map records without
# depending on worksheet order or truncation of the company name.
# Inputs remain unchanged. Every expected company is retained, including
# companies whose Bloomberg responses contain no usable numeric values.

In [39]:
metadata = pd.read_csv(SECURITY_LIST, dtype="string")
required_metadata = {
    "company_id", "company_type", "bloomberg_identifier",
    "prediction_date", "start_date", "end_date",
}
missing = required_metadata - set(metadata.columns)
if missing:
    raise ValueError(f"Security list is missing columns: {sorted(missing)}")
if metadata.empty or metadata["company_id"].isna().any() or metadata["company_id"].duplicated().any():
    raise ValueError("The security list must contain unique, nonmissing company IDs.")
for column in ["prediction_date", "start_date", "end_date", "bankruptcy_date"]:
    if column in metadata:
        metadata[column] = pd.to_datetime(metadata[column], errors="coerce")
if metadata[["prediction_date", "start_date", "end_date"]].isna().any().any():
    raise ValueError("Missing or invalid collection/prediction dates.")
if ((metadata["start_date"] > metadata["end_date"]) |
        (metadata["end_date"] > metadata["prediction_date"])).any():
    raise ValueError("Collection windows must end on or before prediction_date.")
metadata["bankruptcy_label"] = metadata["company_type"].map({"bankrupt": 1, "nonbankrupt": 0})
if metadata["bankruptcy_label"].isna().any():
    raise ValueError("Unknown company_type; expected bankrupt or nonbankrupt.")
metadata["bankruptcy_label"] = metadata["bankruptcy_label"].astype(int)
metadata = metadata.set_index("company_id", drop=False)

workbook_paths = sorted(
    path for path in INPUT_DIR.rglob("*")
    if path.suffix.lower() in {".xlsx", ".xlsm"} and not path.name.startswith("~$")
)
if not workbook_paths:
    raise FileNotFoundError(f"No Excel workbooks found under {INPUT_DIR}")

quality_records = []
company_sources = {}
company_series = {}
macro_series = {}
workbook_summary = []
for path in workbook_paths:
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        relative_path = path.relative_to(ROOT).as_posix()
        found_companies = 0
        if "Security_List" in workbook.sheetnames:
            index_rows = list(workbook["Security_List"].values)
            index = pd.DataFrame(index_rows[1:], columns=index_rows[0]).dropna(subset=["company_id"])
            local_ids = index["company_id"].astype(str).tolist()
            if len(local_ids) != len(set(local_ids)):
                raise ValueError(f"Duplicate Security_List IDs in {path}")
            unknown = set(local_ids) - set(metadata.index)
            if unknown:
                raise ValueError(f"Companies absent from the master security list: {sorted(unknown)}")
            # Stop if the downloaded workbook belongs to an outdated date window.
            for record in index.to_dict("records"):
                company_id = str(record["company_id"])
                for column in ["start_date", "end_date", "prediction_date"]:
                    if source_date(record[column]) != metadata.at[company_id, column]:
                        raise ValueError(f"Metadata date mismatch for {company_id}, {column}, in {path.name}")
                if str(record["bloomberg_identifier"]).strip() != metadata.at[company_id, "bloomberg_identifier"].strip():
                    raise ValueError(f"Bloomberg identifier mismatch for {company_id} in {path.name}")
            for sheet in workbook:
                if sheet.title in {"Security_List", "Macro_Data"}:
                    continue
                matches = [cid for cid in local_ids if sheet.title.startswith(cid + "_") or sheet.title == cid]
                if len(matches) != 1:
                    raise ValueError(f"Cannot uniquely map sheet {sheet.title!r} in {path.name}")
                company_id = matches[0]
                if company_id in company_series:
                    raise ValueError(f"Duplicate input worksheets for {company_id}; remove duplicate workbook copies from data/processed.")
                rows = list(sheet.values)
                context = {"company_id": company_id, "workbook": relative_path, "sheet": sheet.title}
                parsed = {}
                for field in [*ANNUAL_TREND_METHOD, *MARKET_FIELDS]:
                    series = parse_field(rows, field, context, quality_records)
                    in_window = (
                        (series.index >= metadata.at[company_id, "start_date"]) &
                        (series.index <= metadata.at[company_id, "end_date"])
                    )
                    quality_records[-1]["outside_collection_window"] = int((~in_window).sum())
                    parsed[field] = series.loc[in_window]
                company_series[company_id] = parsed
                company_sources[company_id] = {"source_workbook": relative_path, "source_sheet": sheet.title}
                found_companies += 1
        if "Macro_Data" in workbook.sheetnames:
            if macro_series:
                raise ValueError("Multiple Macro_Data worksheets found; keep one authoritative populated macro workbook.")
            rows = list(workbook["Macro_Data"].values)
            for field in MACRO_FIELDS:
                macro_series[field] = parse_field(
                    rows, field, {"company_id": "MACRO", "workbook": relative_path, "sheet": "Macro_Data"},
                    quality_records,
                )
        workbook_summary.append({"workbook": relative_path, "company_sheets": found_companies,
                                 "has_macro": "Macro_Data" in workbook.sheetnames})
    finally:
        workbook.close()
    print(f"Read {path.name}: {found_companies} company sheets")

if not macro_series:
    raise FileNotFoundError("No populated Macro_Data worksheet found under data/processed.")
missing_companies = sorted(set(metadata.index) - set(company_series))
for company_id in missing_companies:
    company_series[company_id] = {
        field: pd.Series(dtype=float, name=field, index=pd.DatetimeIndex([], name="date"))
        for field in [*ANNUAL_TREND_METHOD, *MARKET_FIELDS]
    }
    quality_records.append({"company_id": company_id, "status": "missing_company_sheet"})
if missing_companies:
    warnings.warn(f"{len(missing_companies)} companies have no worksheet; retained with missing features.")
print(f"Loaded {len(metadata)} companies and {len(macro_series)} macro series.")
display(pd.DataFrame(workbook_summary))

Read bloomberg_macro_addin_workbook.xlsx: 0 company sheets
Read bloomberg_addin_workbook_part_001.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_002.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_003.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_004.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_005.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_006.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_007.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_008.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_009.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_010.xlsx: 50 company sheets
Read bloomberg_addin_workbook_part_011.xlsx: 38 company sheets
Loaded 538 companies and 8 macro series.


,workbook,company_sheets,has_macro
0,data/processed/macro/bloomberg_macro_addin_wor...,0,True
1,data/processed/security/bloomberg_addin_workbo...,50,False
2,data/processed/security/bloomberg_addin_workbo...,50,False
3,data/processed/security/bloomberg_addin_workbo...,50,False
4,data/processed/security/bloomberg_addin_workbo...,50,False
5,data/processed/security/bloomberg_addin_workbo...,50,False
6,data/processed/security/bloomberg_addin_workbo...,50,False
7,data/processed/security/bloomberg_addin_workbo...,50,False
8,data/processed/security/bloomberg_addin_workbo...,50,False
9,data/processed/security/bloomberg_addin_workbo...,50,False


In [25]:
# Compute full histories before taking the three lag snapshots
#
# Annual changes compare consecutive fiscal observations separated by
# 300-430 days, so a missing fiscal year never becomes a one-year trend.
# pct_change is a fraction (0.10 = 10%); absolute_change is a signed difference.
# Market windows use a complete monthly index with no missing-month filling.
# Rolling statistics require all observations in their stated window.

In [26]:
annual_by_company = {}
market_by_company = {}
for company_id, record in metadata.iterrows():
    annual_by_company[company_id] = annual_history(company_series[company_id])
    market_by_company[company_id] = monthly_history(
        company_series[company_id], record["start_date"], record["end_date"]
    )

macro_monthly, macro_source_dates = macro_monthly_history(
    macro_series, metadata["start_date"].min(), metadata["prediction_date"].max()
)
print("Annual rows:", sum(len(frame) for frame in annual_by_company.values()))
print("Monthly company rows:", sum(len(frame) for frame in market_by_company.values()))
print("Monthly macro rows:", len(macro_monthly))

Annual rows: 2280
Monthly company rows: 32818
Monthly macro rows: 396


In [27]:
# Assemble one snapshot for each company and lag
#
# Annual values/trends use the latest fiscal observation by the cutoff,
# subject to the publication-delay and staleness settings above.
# Market and macro values use the last COMPLETE month by the same cutoff.
# Financial ratios use annual inputs from that snapshot and market_cap from
# its market month. A zero denominator produces NaN, never infinity.
# Source dates and cutoffs are retained for checking the time alignment.

In [28]:
annual_columns = [
    *ANNUAL_TREND_METHOD,
    *(f"{field}_{method}" for field, method in ANNUAL_TREND_METHOD.items()),
]
market_columns = [
    *MARKET_FIELDS,
    *(f"{field}_pct_change_{window}" for field, windows in MARKET_TREND_FEATURES.items() for window in windows),
    *MARKET_DERIVED_FEATURES,
]
macro_columns = [*MACRO_FIELDS, *MACRO_DERIVED_FEATURES]
feature_names = [*annual_columns, *DERIVED_FINANCIAL_FORMULAS, *market_columns, *macro_columns]
assert len(feature_names) == len(set(feature_names))

snapshot_records = []
for company_id, record in metadata.iterrows():
    annual = annual_by_company[company_id]
    market = market_by_company[company_id]
    for lag, years_before in LAG_OFFSETS_YEARS.items():
        cutoff = record["prediction_date"] - pd.DateOffset(years=years_before)
        month = completed_month(cutoff)
        snapshot = {
            "company_id": company_id, "lag_year": lag,
            "cutoff_date": cutoff, "annual_source_date": pd.NaT,
            "market_month": month.to_timestamp("M"),
            "macro_month": month.to_timestamp("M"),
        }
        values = {field: np.nan for field in feature_names}
        availability = annual.index + pd.to_timedelta(ANNUAL_PUBLICATION_DELAY_DAYS, unit="D")
        eligible = annual.loc[
            (availability <= cutoff) &
            (availability >= cutoff - pd.Timedelta(days=MAX_ANNUAL_AGE_DAYS))
        ]
        if not eligible.empty:
            values.update(eligible.iloc[-1].to_dict())
            snapshot["annual_source_date"] = eligible.index[-1]
        if month in market.index:
            values.update(market.loc[month].to_dict())
        if month in macro_monthly.index:
            values.update(macro_monthly.loc[month].to_dict())
        for field, expression in DERIVED_FINANCIAL_FORMULAS.items():
            values[field] = financial_formula(expression, values)
        snapshot.update(values)
        snapshot_records.append(snapshot)

company_lag_snapshots = pd.DataFrame(snapshot_records)
company_lag_snapshots[feature_names] = company_lag_snapshots[feature_names].astype(float).replace([np.inf, -np.inf], np.nan)
pieces = []
snapshot_info = ["cutoff_date", "annual_source_date", "market_month", "macro_month"]
for lag in LAG_OFFSETS_YEARS:
    piece = company_lag_snapshots.loc[company_lag_snapshots["lag_year"].eq(lag)].set_index("company_id")
    piece = piece[[*snapshot_info, *feature_names]].add_suffix(f"_lag_{lag}y")
    pieces.append(piece)
features_wide = pd.concat(pieces, axis=1).reindex(metadata.index)

# Keep identifying metadata for joins and audit, not as predictor variables.
final_company_dataset = pd.concat(
    [metadata.drop(columns=["bankruptcy_label"]), features_wide,
     metadata[["bankruptcy_label"]]],
    axis=1,
)
final_company_dataset = final_company_dataset.reset_index(drop=True)
model_feature_columns = [f"{field}_lag_{lag}y" for lag in LAG_OFFSETS_YEARS for field in feature_names]

print("Final dataset shape:", final_company_dataset.shape)
print("Requested features per lag:", len(feature_names))
print("Model feature columns:", len(model_feature_columns))
print("Labels:", final_company_dataset["bankruptcy_label"].value_counts().sort_index().to_dict())
display(final_company_dataset[["company_id", "company_type", "total_assets_lag_1y",
                               "total_assets_pct_change_lag_1y", "bankruptcy_label"]].head(10))

Final dataset shape: (538, 405)
Requested features per lag: 127
Model feature columns: 381
Labels: {0: 269, 1: 269}


,company_id,company_type,total_assets_lag_1y,total_assets_pct_change_lag_1y,bankruptcy_label
0,SA_222,bankrupt,222.3550,-0.136814,1
1,SA_187,bankrupt,2005.1270,0.298980,1
2,SA_135,bankrupt,466.5490,-0.112326,1
3,SA_4,bankrupt,228.5830,NaN,1
4,SA_26,bankrupt,317.5870,-0.197220,1
5,SA_172,bankrupt,1563.5630,-0.082480,1
6,SA_80,bankrupt,1951.4611,-0.044725,1
7,SA_134,bankrupt,210.9540,0.086899,1
8,SA_52,bankrupt,213.9520,-0.022264,1
9,SA_145,bankrupt,689.3960,0.537622,1


In [29]:
# Document every feature and report data coverage
#
# Original monetary units and sign conventions are preserved from Bloomberg.
# Ratios assume compatible source units; percentage changes use signed bases.
# Absolute-change features keep their original units and signed direction.
# Labels reflect supplied company membership, not a newly estimated horizon.
# company_type, bankruptcy_date, IDs, and collection dates are audit metadata;
# use model_feature_columns for modeling to avoid label/identifier leakage.

In [30]:
base_definitions = {}
for field, method in ANNUAL_TREND_METHOD.items():
    base_definitions[field] = ("annual_original", "latest fiscal observation at snapshot", "Bloomberg source units")
    formula = "current / previous - 1" if method == "pct_change" else "current - previous"
    units = "fractional change" if method == "pct_change" else "Bloomberg source units"
    base_definitions[f"{field}_{method}"] = ("annual_trend", f"{field}: {formula}; fiscal gap 300-430 days", units)
for field, formula in DERIVED_FINANCIAL_FORMULAS.items():
    base_definitions[field] = ("financial_ratio", formula, "ratio")
for field in MARKET_FIELDS:
    base_definitions[field] = ("market_original", "last observation in the snapshot's complete calendar month", "Bloomberg source units")
for field, windows in MARKET_TREND_FEATURES.items():
    for window, periods in windows.items():
        base_definitions[f"{field}_pct_change_{window}"] = (
            "market_trend", f"{field}[t] / {field}[t - {periods} calendar months] - 1",
            "fractional change",
        )
for field, formula in MARKET_DERIVED_FEATURES.items():
    base_definitions[field] = ("market_derived", formula + "; complete monthly grid, no fill, full rolling windows", "see formula/source units")
for field, config in MACRO_FIELDS.items():
    base_definitions[field] = ("macro_original", f"{config['security']} / {config['field']}; latest observation by month-end", "Bloomberg source units")
for field, formula in MACRO_DERIVED_FEATURES.items():
    base_definitions[field] = ("macro_derived", formula + "; differences use month-end levels", "0/1" if field == "yield_curve_inverted" else "source-unit difference")

dictionary_rows = []
for lag, offset in LAG_OFFSETS_YEARS.items():
    for field in feature_names:
        family, formula, units = base_definitions[field]
        dictionary_rows.append({
            "column": f"{field}_lag_{lag}y", "base_feature": field, "family": family,
            "lag_label": lag, "cutoff_years_before_prediction": offset,
            "formula": formula, "units": units,
            "missing_policy": "NaN retained; zero-denominator ratios and incomplete rolling windows are NaN",
            "source_note": (
                "LUACTRUU PX_LAST is a corporate-bond total-return index, not a yield spread; requested alias retained."
                if "credit_spread" in field else ""
            ),
        })
feature_dictionary = pd.DataFrame(dictionary_rows)
feature_missingness = pd.DataFrame({
    "column": model_feature_columns,
    "missing_count": final_company_dataset[model_feature_columns].isna().sum().to_numpy(),
    "missing_fraction": final_company_dataset[model_feature_columns].isna().mean().to_numpy(),
})
for label in [0, 1]:
    subset = final_company_dataset.loc[final_company_dataset["bankruptcy_label"].eq(label), model_feature_columns]
    feature_missingness[f"missing_fraction_label_{label}"] = subset.isna().mean().to_numpy()
feature_missingness = feature_missingness.sort_values(["missing_fraction", "column"], ascending=[False, True])

quality_by_company = metadata[["company_id", "company_type", "bankruptcy_label"]].copy()
quality_by_company.index.name = None
quality_by_company["source_workbook"] = [company_sources.get(cid, {}).get("source_workbook", "") for cid in metadata.index]
quality_by_company["source_sheet"] = [company_sources.get(cid, {}).get("source_sheet", "") for cid in metadata.index]
quality_by_company["annual_observations"] = [len(annual_by_company[cid]) for cid in metadata.index]
quality_by_company["monthly_price_observations"] = [market_by_company[cid]["adjusted_price"].notna().sum() for cid in metadata.index]
quality_by_company["missing_feature_count"] = final_company_dataset[model_feature_columns].isna().sum(axis=1).to_numpy()
quality_by_company["missing_feature_fraction"] = final_company_dataset[model_feature_columns].isna().mean(axis=1).to_numpy()
for lag in LAG_OFFSETS_YEARS:
    columns = [f"{field}_lag_{lag}y" for field in feature_names]
    quality_by_company[f"missing_fraction_lag_{lag}y"] = final_company_dataset[columns].isna().mean(axis=1).to_numpy()
source_quality_report = pd.DataFrame(quality_records)

display(feature_missingness.head(20))
print("Completely missing feature columns:", int(feature_missingness["missing_fraction"].eq(1).sum()))
print("Source fields without usable numbers:", int(source_quality_report["status"].ne("ok").sum()))
print("No companies dropped. Inspect coverage before training or choosing an imputation policy.")

,column,missing_count,missing_fraction,missing_fraction_label_0,missing_fraction_label_1
84,beta_lag_1y,538,1.000000,1.000000,1.000000
211,beta_lag_2y,538,1.000000,1.000000,1.000000
338,beta_lag_3y,538,1.000000,1.000000,1.000000
87,bid_ask_spread_lag_1y,538,1.000000,1.000000,1.000000
214,bid_ask_spread_lag_2y,538,1.000000,1.000000,1.000000
341,bid_ask_spread_lag_3y,538,1.000000,1.000000,1.000000
350,shares_outstanding_pct_change_3y_lag_3y,363,0.674721,0.356877,0.992565
347,market_cap_pct_change_3y_lag_3y,362,0.672862,0.353160,0.992565
353,volume_pct_change_3y_lag_3y,362,0.672862,0.353160,0.992565
344,adjusted_price_pct_change_3y_lag_3y,361,0.671004,0.349442,0.992565


Completely missing feature columns: 6
Source fields without usable numbers: 2758
No companies dropped. Inspect coverage before training or choosing an imputation policy.


In [31]:
# Verify temporal alignment and save the final dataset
#
# The CSV is one row per company. A long snapshot table, field dictionary,
# source-quality report, and missingness reports make the construction auditable.
# Cleaned history files retain the extra baseline years needed for calculations.

In [32]:
assert len(final_company_dataset) == len(metadata)
assert final_company_dataset["company_id"].is_unique
assert final_company_dataset["bankruptcy_label"].isin([0, 1]).all()
assert set(final_company_dataset["company_id"]) == set(metadata.index)
assert len(company_lag_snapshots) == len(metadata) * len(LAG_OFFSETS_YEARS)
assert not company_lag_snapshots.duplicated(["company_id", "lag_year"]).any()
assert set(feature_dictionary["column"]) == set(model_feature_columns)
assert not np.isinf(final_company_dataset[model_feature_columns].to_numpy(dtype=float)).any()
for lag, years_before in LAG_OFFSETS_YEARS.items():
    cutoff = final_company_dataset[f"cutoff_date_lag_{lag}y"]
    expected = final_company_dataset["prediction_date"].map(lambda date: date - pd.DateOffset(years=years_before))
    assert cutoff.equals(expected.rename(cutoff.name))
    source = final_company_dataset[f"annual_source_date_lag_{lag}y"]
    assert (source.isna() | (source + pd.to_timedelta(ANNUAL_PUBLICATION_DELAY_DAYS, unit="D") <= cutoff)).all()
    assert (final_company_dataset[f"market_month_lag_{lag}y"] <= cutoff).all()
    assert (final_company_dataset[f"macro_month_lag_{lag}y"] <= cutoff).all()
for field in MACRO_FIELDS:
    available = macro_source_dates[field] + pd.to_timedelta(MACRO_PUBLICATION_DELAY_DAYS[field], unit="D")
    assert (available.isna() | (available <= macro_source_dates.index.to_timestamp("M"))).all()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_company_dataset.to_csv(OUTPUT_DIR / "final_company_dataset.csv", index=False)
company_lag_snapshots.to_csv(OUTPUT_DIR / "company_lag_snapshots.csv", index=False)
feature_dictionary.to_csv(OUTPUT_DIR / "feature_dictionary.csv", index=False)
feature_missingness.to_csv(OUTPUT_DIR / "feature_missingness_report.csv", index=False)
quality_by_company.to_csv(OUTPUT_DIR / "company_quality_report.csv", index=False)
source_quality_report.to_csv(OUTPUT_DIR / "source_quality_report.csv", index=False)

annual_export = pd.concat(annual_by_company, names=["company_id", "date"]).reset_index()
annual_export.to_csv(OUTPUT_DIR / "cleaned_annual_history.csv", index=False)
market_export = pd.concat(market_by_company, names=["company_id", "month"]).reset_index()
market_export["month"] = market_export["month"].dt.to_timestamp("M")
market_export.to_csv(OUTPUT_DIR / "cleaned_market_monthly.csv", index=False)
macro_export = macro_monthly.copy()
macro_export.index = macro_export.index.to_timestamp("M")
macro_export.to_csv(OUTPUT_DIR / "cleaned_macro_monthly.csv", index_label="date")
macro_dates_export = macro_source_dates.copy()
macro_dates_export.index = macro_dates_export.index.to_timestamp("M")
macro_dates_export.to_csv(OUTPUT_DIR / "macro_source_dates.csv", index_label="month_end")

manifest = {
    "security_list": SECURITY_LIST.relative_to(ROOT).as_posix(),
    "workbooks": [path.relative_to(ROOT).as_posix() for path in workbook_paths],
    "company_count": len(final_company_dataset),
    "bankruptcy_label_counts": {str(k): int(v) for k, v in final_company_dataset["bankruptcy_label"].value_counts().items()},
    "model_feature_columns": model_feature_columns,
    "lag_offsets_years": LAG_OFFSETS_YEARS,
    "annual_publication_delay_days": ANNUAL_PUBLICATION_DELAY_DAYS,
    "macro_publication_delay_days": MACRO_PUBLICATION_DELAY_DAYS,
    "max_annual_age_days": MAX_ANNUAL_AGE_DAYS,
    "annual_gap_days": [MIN_ANNUAL_GAP_DAYS, MAX_ANNUAL_GAP_DAYS],
    "market_policy": "Complete calendar months; no missing-month fill; full rolling windows.",
    "point_in_time_warning": "Recorded observation dates are not verified publication dates. Historical revisions may remain.",
    "credit_spread_warning": "LUACTRUU PX_LAST is a total-return index; requested credit_spread alias retained.",
    "long_window_warning": "A 36-month trend at the oldest lag may lack its baseline in a five-year download; it remains NaN.",
}
(OUTPUT_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("Saved:", OUTPUT_DIR / "final_company_dataset.csv")
print(f"{len(final_company_dataset)} companies; {len(model_feature_columns)} model feature columns; all checks passed.")

Saved: C:\Users\MyPC\myprojects\bankruptcy-prediction\data\clean\final_company_dataset.csv
538 companies; 381 model feature columns; all checks passed.
